# OCR, Layout Analysis & Table Extraction — Turning Documents into Data

> **Orbit 5 (Multimodal)** · Notebook 6 of 22

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/multimodal/06_ocr_layout_and_table_extraction.ipynb)

## Learning Objectives

By the end of this notebook you will be able to:

- **Understand why raw OCR loses critical document structure** — characters without layout are just a bag of words, stripping away the meaning carried by spatial relationships.
- **Build a VLM-based document understanding pipeline with Qwen2.5-VL** that performs OCR, layout analysis, and structured extraction in a single forward pass.
- **Compare VLM zero-shot OCR against dedicated OCR engines (EasyOCR)** to understand the speed-vs-understanding trade-off.
- **Extract structured tables from document images into DataFrames** using JSON-targeted prompting, with arithmetic validation of the results.
- **Implement layout-aware prompting strategies** that target specific document zones — headers, paragraphs, tables, and footnotes — for surgical information retrieval.

## 1 · Why OCR Is Still Necessary

Despite the rise of born-digital documents, an enormous volume of the world's information still lives as **scanned PDFs, photographed whiteboards, handwritten notes, and legacy paper records**. Hospitals scan intake forms. Governments archive decades of printed legislation. Warehouses photograph shipping labels. Every one of these images contains text that machines cannot natively search, sort, or analyse.

Traditional OCR engines — Tesseract, EasyOCR, PaddleOCR — solve the character-recognition problem admirably: they convert pixel patterns into Unicode strings. But that conversion **discards all layout information**. A two-column academic paper becomes a single stream of interleaved text. A table's rows and columns collapse into an unstructured paragraph. The number "$4,285.75" loses its association with the label "TOTAL DUE."

This is where **Vision-Language Models (VLMs)** change the game. A VLM sees the page the way a human does: it perceives spatial relationships between text blocks and understands that a number sitting beneath the word "Total" carries a fundamentally different meaning than the same number beside "Quantity." The model performs OCR *and* layout analysis *and* semantic interpretation in one pass.

In this notebook we use **Qwen2.5-VL-7B** as a zero-shot document-understanding engine and benchmark it against EasyOCR on synthetic invoice and receipt images.

## 2 · The OCR Pipeline — From Pixels to Structured Text

A traditional OCR system is a multi-stage pipeline:

```
Image → Pre-processing → Text Detection → Text Recognition → Post-processing → Plain Text
         (binarise,        (find text        (read chars       (spell-check,
          deskew,           regions /          in each           merge lines)
          denoise)          bounding boxes)    region)
```

Layout analysis inserts an additional step after text detection — **region classification** — that labels each detected zone as *header*, *paragraph*, *table*, *figure caption*, or *footnote*. Tools like LayoutLMv3 and Donut were specifically designed for this classification task.

The modern **VLM approach collapses every stage into a single model call**: the image is encoded by a vision transformer, the prompt specifies what to extract, and the language model decodes the answer in whatever structured format we request — plain text, Markdown, or JSON.

| Dimension | Traditional OCR | VLM-based OCR |
|-----------|----------------|---------------|
| Speed | Fast (~100 ms/page) | Slower (~2-10 s/page) |
| Layout understanding | None (or separate model) | Built-in |
| Output format | Raw text + bounding boxes | Any format via prompting |
| Accuracy on clean text | Very high | High, but may hallucinate |
| Training required | Language-specific models | Zero-shot |

The practical takeaway: **use dedicated OCR when you need speed and raw text; use VLMs when you need understanding.**

## 3 · Environment Setup

In [ ]:
!pip install -q "transformers>=4.57.0" accelerate bitsandbytes torch pillow qwen-vl-utils easyocr requests pandas

In [ ]:
import torch, json, time
from PIL import Image, ImageDraw, ImageFont
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
import easyocr
import pandas as pd
import numpy as np

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map="auto"
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
ocr_reader = easyocr.Reader(["en"], gpu=torch.cuda.is_available())

print(f"✓ VLM loaded: {MODEL_ID}  |  VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB")
print(f"✓ EasyOCR reader ready")

### Helper — `ask_vlm`

A thin wrapper that sends an image and a text prompt to Qwen2.5-VL and returns the decoded response string. Identical to the helper in notebook 05 so that all multimodal notebooks share a consistent calling convention.

In [ ]:
from qwen_vl_utils import process_vision_info

def ask_vlm(image: Image.Image, prompt: str, max_tokens: int = 1024) -> str:
    """Send an image + prompt to Qwen2.5-VL and return the text response."""
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt},
        ],
    }]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        ids = model.generate(**inputs, max_new_tokens=max_tokens)
    ids_trimmed = ids[:, inputs.input_ids.shape[1]:]
    return processor.batch_decode(ids_trimmed, skip_special_tokens=True)[0]

## 4 · Creating Synthetic Document Images

Rather than relying on external datasets, we **render our own documents with PIL**. This gives us full control over the ground truth — we know exactly what text appears and where, making it trivial to validate extraction accuracy.

In [ ]:
def make_invoice_image():
    img = Image.new("RGB", (800, 1000), "white")
    draw = ImageDraw.Draw(img)

    # Header
    draw.text((50, 30), "ACME CORPORATION", fill="black")
    draw.text((50, 55), "Invoice #INV-2024-0847", fill="darkblue")
    draw.line([(50, 80), (750, 80)], fill="black", width=2)

    # Addresses
    draw.text((50, 100), "Bill To:", fill="gray")
    draw.text((50, 120), "TechStart Inc.", fill="black")
    draw.text((50, 140), "456 Innovation Drive", fill="black")
    draw.text((50, 160), "San Francisco, CA 94107", fill="black")
    draw.text((420, 100), "Ship To:", fill="gray")
    draw.text((420, 120), "TechStart Warehouse", fill="black")
    draw.text((420, 140), "789 Commerce Blvd", fill="black")
    draw.text((420, 160), "Oakland, CA 94612", fill="black")

    # Table header
    y = 220
    draw.line([(50, y), (750, y)], fill="black", width=2)
    cols = [50, 300, 450, 580, 700]
    headers = ["Item", "Qty", "Unit Price", "Total"]
    for i, h in enumerate(headers):
        draw.text((cols[i], y + 5), h, fill="black")
    y += 30
    draw.line([(50, y), (750, y)], fill="gray", width=1)

    # Table rows
    rows = [
        ("Cloud Hosting (Annual)", "1", "$2,400.00", "$2,400.00"),
        ("API Gateway License", "3", "$150.00", "$450.00"),
        ("Premium Support", "1", "$600.00", "$600.00"),
        ("Data Transfer (10TB)", "10", "$50.00", "$500.00"),
    ]
    for item, qty, price, total in rows:
        draw.text((cols[0], y + 5), item, fill="black")
        draw.text((cols[1], y + 5), qty, fill="black")
        draw.text((cols[2], y + 5), price, fill="black")
        draw.text((cols[3], y + 5), total, fill="black")
        y += 28
        draw.line([(50, y), (750, y)], fill="lightgray", width=1)

    # Totals
    y += 15
    draw.line([(450, y), (750, y)], fill="black", width=2)
    draw.text((450, y + 8), "Subtotal:", fill="black")
    draw.text((cols[3], y + 8), "$3,950.00", fill="black")
    y += 28
    draw.text((450, y), "Tax (8.5%):", fill="black")
    draw.text((cols[3], y), "$335.75", fill="black")
    y += 28
    draw.text((450, y), "TOTAL DUE:", fill="black")
    draw.text((cols[3], y), "$4,285.75", fill="darkblue")

    # Footnote
    draw.text((50, 900), "Payment terms: Net 30. Late payments subject to 1.5% monthly interest.", fill="gray")
    draw.text((50, 920), "Questions? billing@acmecorp.example.com", fill="gray")
    return img

invoice = make_invoice_image()
invoice

## 5 · VLM as Zero-Shot OCR Engine

Instead of building a multi-stage OCR pipeline, we hand the entire document image to Qwen2.5-VL with a single prompt. The model reads the text **and** understands the layout simultaneously — it knows that "Bill To:" is a section label, that "TechStart Inc." is the corresponding value, and that the four-column grid is a table.

This is zero-shot document understanding: no fine-tuning, no template matching, no custom rules. The same model and prompt pattern works on invoices, receipts, contracts, and handwritten notes.

The key advantage is **flexibility** — a single model replaces an entire pipeline. The key limitation is **speed**: a VLM takes several seconds per page where dedicated OCR finishes in milliseconds. VLMs can also **hallucinate** text that is not present in the image, so validation is essential.

In [ ]:
print("═══ VLM Full-Page OCR ═══")
t0 = time.time()
response = ask_vlm(
    invoice,
    "Read all text in this document image, preserving the layout structure. "
    "Group text by sections (header, addresses, table, totals, footnotes)."
)
vlm_time = time.time() - t0
print(response)
print(f"\n⏱ VLM OCR took {vlm_time:.1f}s")

In [ ]:
print("═══ EasyOCR Output ═══")
t0 = time.time()
results = ocr_reader.readtext(np.array(invoice))
easyocr_time = time.time() - t0

for bbox, text, conf in results:
    print(f"  [{conf:.2f}] {text}")

print(f"\nTotal text regions detected: {len(results)}")
print(f"⏱ EasyOCR took {easyocr_time:.2f}s")
print("Note: EasyOCR gives us text + bounding boxes, but NO structural understanding.")

### Comparison Analysis

The two approaches reveal a clear trade-off. **EasyOCR** returned every text region with high character-level accuracy and bounding-box coordinates — but it has no concept of sections, tables, or labels. The results are a flat list of text fragments in reading order. You would need substantial post-processing to reconstruct the invoice structure.

**Qwen2.5-VL** returned the text grouped by semantic sections — header, addresses, line items, totals — without any custom rules. It understood that the four columns form a table and that "TOTAL DUE" is the final amount. However, it took significantly longer and may occasionally paraphrase or reformat the original text.

## 6 · Table Extraction — The Hard Problem

Tables are the **single most valuable extraction target** in business documents. Invoices, financial statements, lab reports, and government filings all encode their most critical data in tabular form.

Why are tables hard to extract?

- **Complex headers**: merged cells, multi-row headers, rotated text.
- **Variable formats**: some tables use visible gridlines, others rely on whitespace alignment, and some mix both.
- **Implicit structure**: a "totals" row may have a different column span than the data rows above it.

We will explore three prompting strategies, ordered by increasing utility for downstream code:

1. **Raw text** — ask the model to simply read the table. Easy to produce, hard to parse programmatically.
2. **Markdown table** — a structured but human-readable format. Good for display, moderate for parsing.
3. **JSON** — the gold standard for machine consumption. Enables direct conversion to a DataFrame.

After extraction, **always validate**: do the individual line totals match quantity × unit price? Does the column sum match the stated subtotal? Arithmetic cross-checks catch hallucinations that look plausible but are numerically wrong.

In [ ]:
print("═══ Table Extraction: Markdown Format ═══")
response = ask_vlm(
    invoice,
    "Extract the table from this invoice as a markdown table. "
    "Include all rows and columns exactly as they appear."
)
print(response)

In [ ]:
print("═══ Table Extraction: JSON Format ═══")
prompt = """Extract the line items table from this invoice as JSON.
Use this exact schema:
{"items": [{"description": "...", "quantity": N, "unit_price": N.NN, "total": N.NN}]}
Output ONLY valid JSON, no explanation."""

response = ask_vlm(invoice, prompt)
print(response)

try:
    data = json.loads(response)
    df = pd.DataFrame(data["items"])
    print("\nParsed DataFrame:")
    print(df.to_string(index=False))
    print(f"\nComputed sum: ${df['total'].sum():.2f}")
except json.JSONDecodeError as e:
    print(f"\nJSON parse failed: {e}")
    print("This is common — VLMs don't always produce valid JSON on the first try.")

### Arithmetic Validation

We know the ground truth because we rendered the image ourselves. Let us cross-check the extracted values against the known data.

In [ ]:
# Ground-truth line items from our rendering code
ground_truth = [
    {"description": "Cloud Hosting (Annual)", "quantity": 1, "unit_price": 2400.00, "total": 2400.00},
    {"description": "API Gateway License",    "quantity": 3, "unit_price": 150.00,  "total": 450.00},
    {"description": "Premium Support",         "quantity": 1, "unit_price": 600.00,  "total": 600.00},
    {"description": "Data Transfer (10TB)",    "quantity": 10, "unit_price": 50.00,  "total": 500.00},
]
gt_df = pd.DataFrame(ground_truth)

print("Ground-truth table:")
print(gt_df.to_string(index=False))
print(f"\nSubtotal: ${gt_df['total'].sum():.2f}")
print(f"Tax (8.5%): ${gt_df['total'].sum() * 0.085:.2f}")
print(f"Grand total: ${gt_df['total'].sum() * 1.085:.2f}")

# Validate each row: quantity × unit_price == total
gt_df["check"] = gt_df["quantity"] * gt_df["unit_price"]
gt_df["valid"] = gt_df["check"] == gt_df["total"]
print(f"\nRow-level validation: {gt_df['valid'].all()} (all rows pass)")

## 7 · Layout Understanding — Semantic Zones

A document is not a flat stream of text — it is a collection of **semantic zones**. Headers establish context, body paragraphs carry content, tables encode structured data, and footnotes provide caveats or legal fine print. A VLM can classify and target these zones when prompted correctly.

Layout-aware extraction means asking **specific questions** rather than a blanket "read everything." The prompt "What is the invoice number?" will yield a more precise answer than parsing the full OCR output and searching for a pattern. For multi-page documents, process each page individually and merge the results — VLMs currently work best on single-page inputs.

This approach also enables **field-level confidence**: if two differently-worded prompts return the same invoice number, confidence is high; if they disagree, the extraction should be flagged for human review.

In [ ]:
print("═══ Targeted Layout Extraction ═══\n")

questions = [
    ("Invoice Number",   "What is the invoice number in this document?"),
    ("Billing Address",  "What is the full 'Bill To' address?"),
    ("Shipping Address", "What is the full 'Ship To' address?"),
    ("Total Amount",     "What is the total amount due?"),
    ("Payment Terms",    "What are the payment terms?"),
]

for label, q in questions:
    resp = ask_vlm(invoice, q + " Reply concisely.", max_tokens=100)
    print(f"{label:20s}: {resp.strip()}")

## 8 · A Second Document — Coffee Receipt

To verify that our pipeline generalises, we create a completely different document type: a small retail receipt. The layout is narrower, the items are simpler, and there is no formal table grid — just whitespace-aligned text.

In [ ]:
def make_receipt_image():
    img = Image.new("RGB", (400, 600), "white")
    draw = ImageDraw.Draw(img)

    draw.text((120, 20), "COFFEE HOUSE", fill="black")
    draw.text((130, 45), "123 Bean Street", fill="gray")
    draw.line([(30, 70), (370, 70)], fill="black")

    items = [("Latte", "$4.50"), ("Croissant", "$3.25"),
             ("Espresso", "$3.00"), ("Muffin", "$2.75")]
    y = 85
    for item, price in items:
        draw.text((40, y), item, fill="black")
        draw.text((300, y), price, fill="black")
        y += 25

    draw.line([(30, y + 5), (370, y + 5)], fill="black")
    draw.text((200, y + 15), "Subtotal: $13.50", fill="black")
    draw.text((200, y + 35), "Tax:       $1.15", fill="black")
    draw.text((200, y + 55), "Total:    $14.65", fill="black")
    draw.text((80, y + 90), "Thank you! Come again!", fill="gray")
    return img

receipt = make_receipt_image()
receipt

In [ ]:
print("═══ Receipt Extraction (VLM) ═══")
receipt_text = ask_vlm(
    receipt,
    "Read this receipt. List every item with its price, "
    "then state the subtotal, tax, and total."
)
print(receipt_text)

In [ ]:
print("═══ Receipt Extraction (EasyOCR) ═══")
receipt_results = ocr_reader.readtext(np.array(receipt))
for bbox, text, conf in receipt_results:
    print(f"  [{conf:.2f}] {text}")
print(f"\nText regions: {len(receipt_results)}")

## 9 · Building a Robust Extraction Pipeline

In production, a single model call is never enough. Real-world documents are noisy, skewed, partially occluded, and come in thousands of formats. A robust pipeline needs:

- **Retry logic**: if JSON extraction fails, re-prompt with a simplified schema or a different phrasing.
- **Cross-validation**: extract the same field with two different prompts and compare. Agreement means high confidence; disagreement triggers human review.
- **Arithmetic checks**: for financial documents, verify that extracted quantities × prices equal the stated totals.
- **Confidence estimation**: ask the model to self-assess, or compare the extracted total against an OCR-derived total.

The function below demonstrates a lightweight version of this pattern: it runs a primary extraction and then asks targeted validation questions to cross-check the results.

In [ ]:
def extract_with_validation(image, extraction_prompt, validation_questions):
    """Run primary extraction then cross-check with targeted questions."""
    raw = ask_vlm(image, extraction_prompt)
    validations = {}
    for key, q in validation_questions.items():
        v = ask_vlm(image, q + " Reply in one line.", max_tokens=80)
        validations[key] = v.strip()
    return {"extraction": raw, "validations": validations}

result = extract_with_validation(
    invoice,
    'Extract all line items as JSON: {"items": [{"item": "...", "total": N.NN}]}',
    {
        "total": "What is the grand total on this invoice?",
        "item_count": "How many line items are in the table?",
    }
)

print("Extraction:", result["extraction"][:300], "...")
print("\nValidations:", result["validations"])

## 10 · Exercises

**Exercise 1 — Complex document layout.** Create a PIL image with a multi-column layout (e.g., a two-column newsletter or a form with nested tables). Use `ask_vlm` to extract the text from each column separately and then merge the results. How well does the VLM handle column boundaries?

**Exercise 2 — OCR engine comparison.** Generate three different document types (invoice, receipt, and a simple form with checkboxes). Run both EasyOCR and the VLM on each. For each document, pick five key fields and score extraction accuracy. Which engine wins on which document type?

**Exercise 3 — Retry with rephrasing.** Implement a function `extract_json_with_retry(image, prompts, max_retries=3)` that attempts JSON extraction, catches `json.JSONDecodeError`, and retries with a progressively simpler prompt (e.g., fewer fields, explicit examples). Track how many retries each document needs.

In [ ]:
# ── Exercise 3 starter code ──

def extract_json_with_retry(image, prompts, max_retries=3):
    """
    Try a sequence of prompts until valid JSON is returned.
    `prompts` is a list of strings, ordered from most detailed to simplest.
    """
    for i, prompt in enumerate(prompts[:max_retries]):
        print(f"Attempt {i+1}/{max_retries}...")
        response = ask_vlm(image, prompt)
        try:
            data = json.loads(response)
            print(f"  ✓ Valid JSON on attempt {i+1}")
            return data
        except json.JSONDecodeError:
            print(f"  ✗ Invalid JSON, retrying...")
    print("All retries exhausted.")
    return None

# Example usage (uncomment to run):
# prompts = [
#     'Extract items as JSON: {"items": [{"desc": "...", "qty": N, "price": N.NN, "total": N.NN}]}',
#     'List items and totals as JSON: {"items": [{"name": "...", "total": N.NN}]}',
#     'Return a JSON array of item names: ["item1", "item2"]',
# ]
# result = extract_json_with_retry(invoice, prompts)

## Key Takeaways

1. **VLMs collapse the entire OCR pipeline into a single prompt** — but accuracy varies with document complexity and font size.
2. **Structured prompting (JSON, Markdown table) dramatically outperforms open-ended "read this" prompts** because it constrains the output format and reduces hallucination.
3. **Table extraction is the highest-value task**: always validate with arithmetic cross-checks (quantity × price = line total; sum of line totals = subtotal).
4. **Dedicated OCR (EasyOCR) is faster and more reliable for pure text extraction**; VLMs win when you need layout understanding and semantic interpretation.
5. **Production pipelines need validation, retry logic, and confidence estimation** — never trust a single extraction pass for high-stakes documents.

## References

- **Qwen2.5-VL Technical Report** (2024) — [arXiv:2502.13923](https://arxiv.org/abs/2502.13923)
- **EasyOCR** — [github.com/JaidedAI/EasyOCR](https://github.com/JaidedAI/EasyOCR)
- **LayoutLMv3: Pre-training for Document AI with Unified Text and Image Masking** — Huang et al. (2022)
- **Donut: Document Understanding Transformer without OCR** — Kim et al. (2022)
- Castalia [multimodal/05 — Image Prompting & Visual Reasoning](05_image_prompting_and_visual_reasoning.ipynb)

---

[← 05 Image Prompting & Visual Reasoning](05_image_prompting_and_visual_reasoning.ipynb) | [07 Structured Outputs, Grounding & Localization →](07_structured_outputs_grounding_and_localization.ipynb)